In [ ]:
%pip install numpy
%pip install pandas
%pip install matplotlib
%pip install seaborn
%pip install pathlib
%pip install scikit-learn
%pip install imbalanced-learn
%pip install scipy

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

In [2]:
import os
import multiprocessing

num_cores = multiprocessing.cpu_count()
print(f"La máquina tiene {num_cores} núcleos disponibles.")
hilos_optimos = str(min(num_cores, 64)) 
print(f"Configurando OpenBLAS para usar {hilos_optimos} hilos máximos...")
os.environ['OPENBLAS_NUM_THREADS'] = hilos_optimos
os.environ['MKL_NUM_THREADS'] = hilos_optimos
os.environ['OMP_NUM_THREADS'] = hilos_optimos

La máquina tiene 224 núcleos disponibles.
Configurando OpenBLAS para usar 64 hilos máximos...


In [3]:
import numpy as np
import pandas as pd
import itertools
import json
from pathlib import Path
from collections import Counter
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif, SelectKBest
from sklearn.svm import SVC
from sklearn.metrics import (f1_score, accuracy_score, recall_score,classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.combine import SMOTEENN


In [4]:
X = pd.read_pickle("Sets_Xy/X.pkl")
y = pd.read_pickle("Sets_Xy/y.pkl")

In [5]:
from sklearn.model_selection import train_test_split

#Division estratificada para muestras de cada clase a nivel de cada subset

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, 
    random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, 
    random_state=42)

In [6]:
#Encoding de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)
y_test = le.transform(y_test)

mapeo_labels = pd.DataFrame({
    "label_original": le.classes_,
    "label_encoded": range(len(le.classes_))
})
print("Mapeo de etiquetas:\n", mapeo_labels)
class_names = le.classes_

Mapeo de etiquetas:
                label_original  label_encoded
0                      BENIGN              0
1                         Bot              1
2                        DDoS              2
3               DoS GoldenEye              3
4                    DoS Hulk              4
5            DoS Slowhttptest              5
6               DoS slowloris              6
7                 FTP-Patator              7
8                  Heartbleed              8
9                Infiltration              9
10                   PortScan             10
11                SSH-Patator             11
12    Web Attack  Brute Force             12
13  Web Attack  Sql Injection             13
14            Web Attack  XSS             14


In [7]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import ExtraTreesClassifier

class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, method='tree', k=50, class_weight='balanced', random_state=42):
        self.method = method
        self.k = k
        self.class_weight = class_weight
        self.random_state = random_state
        self.selector = None
        self.feature_names_ = None
        self.support_mask_ = None

    def fit(self, X, y):
        self.feature_names_ = X.columns.tolist() if hasattr(X, 'columns') else list(range(X.shape[1]))
        
        X_array = X.values if hasattr(X, 'values') else X
        y_array = y.values if hasattr(y, 'values') else y
        
        total_features = X_array.shape[1]
        actual_k = min(self.k, total_features)
        
        if self.method == 'none':
            self.support_mask_ = np.ones(total_features, dtype=bool)
            return self

        if self.method == 'f_classif':
            self.selector = SelectKBest(score_func=f_classif, k=actual_k)
            self.selector.fit(X_array, y_array)
            self.support_mask_ = self.selector.get_support()

        elif self.method == 'tree':
            estimator = ExtraTreesClassifier(
                n_estimators=100,               
                max_depth=20,                   
                class_weight=self.class_weight,
                criterion='gini',               
                random_state=self.random_state, 
                n_jobs=-1
            )
            estimator.fit(X_array, y_array)
            importances = estimator.feature_importances_
            
            top_k_indices = np.argsort(importances)[-actual_k:]
            
            self.support_mask_ = np.zeros(total_features, dtype=bool)
            self.support_mask_[top_k_indices] = True

        return self

    def transform(self, X):
        if self.method == 'none':
            return X
            
        X_array = X.values if hasattr(X, 'values') else X
        X_transformed = X_array[:, self.support_mask_]

        if hasattr(X, 'columns'):
            selected_features = np.array(self.feature_names_)[self.support_mask_].tolist()
            return pd.DataFrame(X_transformed, columns=selected_features, index=X.index)
        else:
            return X_transformed

In [8]:
import joblib

X_train_none_2 = joblib.load("Sets_Oversampling_2/X_train_none.joblib")
y_train_none_2 = joblib.load("Sets_Oversampling_2/y_train_none.joblib")

X_train_smote_2 = joblib.load("Sets_Oversampling_2/X_train_smote.joblib")
y_train_smote_2 = joblib.load("Sets_Oversampling_2/y_train_smote.joblib")

X_train_smote_enn_2 = joblib.load("Sets_Oversampling_2/X_train_smote_enn.joblib")
y_train_smote_enn_2 = joblib.load("Sets_Oversampling_2/y_train_smote_enn.joblib")

X_train_smote_tomek_2 = joblib.load("Sets_Oversampling_2/X_train_smote_tomek.joblib")
y_train_smote_tomek_2 = joblib.load("Sets_Oversampling_2/y_train_smote_tomek.joblib")

X_val = pd.read_pickle("Sets_Post_Scaled_Imp/X_val_scaled_bayesian.pkl")
y_val = pd.DataFrame(y_val)

X_test = pd.read_pickle("Sets_Post_Scaled_Imp/X_test_scaled_bayesian.pkl")
y_test = pd.DataFrame(y_test)


In [9]:
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

def balanced_class_weight(y):
    classes = np.unique(y)
    weights = compute_class_weight('balanced', classes=classes, y=y)
    return dict(zip(classes, weights))

def polynomial_class_weight(y, alpha=0.25):
    classes, counts = np.unique(y, return_counts=True)
    weights = 1.0 / np.power(counts, alpha)
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def effective_samples_class_weight(y, beta=0.999):
    classes, counts = np.unique(y, return_counts=True)
    effective_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / effective_num
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def focal_class_weight_improved(y, gamma=2.0):
    classes, counts = np.unique(y, return_counts=True)
    probs = counts / np.sum(counts)
    weights = (1 - probs) ** gamma / probs
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

In [10]:
y_val_1d = y_val.values.ravel() if isinstance(y_val, pd.DataFrame) else y_val.ravel()
y_test_1d = y_test.values.ravel() if isinstance(y_test, pd.DataFrame) else y_test.ravel()

train_datasets = {
    'none': (X_train_none_2, y_train_none_2.values.ravel() if isinstance(y_train_none_2, pd.DataFrame) else y_train_none_2.ravel()),
    'smote': (X_train_smote_2, y_train_smote_2.values.ravel() if isinstance(y_train_smote_2, pd.DataFrame) else y_train_smote_2.ravel()),
    'smote_enn': (X_train_smote_enn_2, y_train_smote_enn_2.values.ravel() if isinstance(y_train_smote_enn_2, pd.DataFrame) else y_train_smote_enn_2.ravel()),
    'smote_tomek': (X_train_smote_tomek_2, y_train_smote_tomek_2.values.ravel() if isinstance(y_train_smote_tomek_2, pd.DataFrame) else y_train_smote_tomek_2.ravel())
}

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
import optuna

dir_name = "Logs_RF_Baseline_1"
os.makedirs(dir_name, exist_ok=True)

def save_confusion_matrix(y_true, y_pred, trial_number, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - Trial {trial_number} ({phase})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'{dir_name}/Trial_{trial_number}_RF_{phase}_CM.png')
    plt.close()

def objective_rf(trial):
    X_train_raw, y_train_raw = train_datasets['none']
    
    rf_params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 10, 100),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'n_jobs': -1,
        'random_state': 42
    }
    
    model = RandomForestClassifier(**rf_params)
    model.fit(X_train_raw, y_train_raw)
    
    y_pred = model.predict(X_val)
    
    f1_mac = f1_score(y_val_1d, y_pred, average='macro')
    report = classification_report(y_val_1d, y_pred, zero_division=0)
    
    save_confusion_matrix(y_val_1d, y_pred, trial.number, phase="Val")
    
    log_msg = f"""Trial {trial.number}
Experimento: Baseline Absoluto con Random Forest
Params RF: {trial.params}

Metricas Clave
F1 Macro (Validation): {f1_mac:.4f}

Classification Report (Precision, Recall, F1 por clase)
{report}
"""
    
    log_filename = f"{dir_name}/Trial_{trial.number}_Metrics.log"
    with open(log_filename, 'w', encoding='utf-8') as f:
        f.write(log_msg)
    
    print(f"Trial {trial.number} completado | F1 Macro: {f1_mac:.4f}")
    
    return f1_mac

print("Iniciando búsqueda con Optuna para Random Forest (Baseline)...")
study_rf = optuna.create_study(direction='maximize', study_name="RandomForest_Baseline_Optimization")

study_rf.optimize(objective_rf, n_trials=100)

best_log_msg = f"""mejor trial para dataset
trial numero: {study_rf.best_trial.number}
f1 macro alcanzado: {study_rf.best_value:.4f}
hiperparametros:
{study_rf.best_params}
"""
with open(f"{dir_name}/mejor_trial.log", 'w') as f:
    f.write(best_log_msg)
    
print(f"Mejor Trial: {study_rf.best_trial.number}")
print(f"Mejor F1 Macro: {study_rf.best_value:.4f}")
print(f"Mejores Hiperparametros:\n{study_rf.best_params}")

[I 2026-03-29 01:00:03,260] A new study created in memory with name: RandomForest_Baseline_Optimization


Iniciando búsqueda con Optuna para Random Forest (Baseline)...


[I 2026-03-29 01:00:28,401] Trial 0 finished with value: 0.687695848525815 and parameters: {'n_estimators': 273, 'max_depth': 67, 'min_samples_split': 20, 'min_samples_leaf': 20, 'max_features': 'log2', 'criterion': 'entropy'}. Best is trial 0 with value: 0.687695848525815.


Trial 0 completado | F1 Macro: 0.6877


[I 2026-03-29 01:01:00,032] Trial 1 finished with value: 0.6869166703111017 and parameters: {'n_estimators': 215, 'max_depth': 76, 'min_samples_split': 5, 'min_samples_leaf': 20, 'max_features': 'sqrt', 'criterion': 'gini'}. Best is trial 0 with value: 0.687695848525815.


...

Trial 97 completado | F1 Macro: 0.8092


[I 2026-03-29 05:52:55,428] Trial 98 finished with value: 0.8274137131014504 and parameters: {'n_estimators': 318, 'max_depth': 98, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': None, 'criterion': 'entropy'}. Best is trial 52 with value: 0.8399577362863335.


Trial 98 completado | F1 Macro: 0.8274


[I 2026-03-29 05:56:39,956] Trial 99 finished with value: 0.8373646951846272 and parameters: {'n_estimators': 335, 'max_depth': 87, 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_features': None, 'criterion': 'entropy'}. Best is trial 52 with value: 0.8399577362863335.


Trial 99 completado | F1 Macro: 0.8374
Mejor Trial: 52
Mejor F1 Macro: 0.8400
Mejores Hiperparametros:
{'n_estimators': 314, 'max_depth': 83, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': None, 'criterion': 'entropy'}


In [12]:
def save_confusion_matrix(y_true, y_pred, trial_number, weight_method, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'CM - {weight_method.upper()} - Trial {trial_number} ({phase})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'{dir_name}/{weight_method}_Trial_{trial_number}_RF_{phase}_CM.png')
    plt.close()

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
import optuna

dir_name = "Logs_RF_Baseline_2"
os.makedirs(dir_name, exist_ok=True)

datasets_to_evaluate = ['smote', 'smote_enn', 'smote_tomek']

for dataset_name in datasets_to_evaluate:
    print(f"Experimento con Dataset: {dataset_name.upper()}")
    
    X_train_current, y_train_current = train_datasets[dataset_name]
    
    def objective_rf(trial):
        rf_params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'max_depth': trial.suggest_int('max_depth', 10, 100),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
            'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
            'n_jobs': -1,
            'random_state': 42
        }
        
        model = RandomForestClassifier(**rf_params)
        model.fit(X_train_current, y_train_current)
        
        y_pred = model.predict(X_val)
        
        f1_mac = f1_score(y_val_1d, y_pred, average='macro')
        report = classification_report(y_val_1d, y_pred, zero_division=0)
        
        save_confusion_matrix(y_val_1d, y_pred, trial.number, dataset_name, phase="Val")
        
        log_msg = f"""Trial {trial.number}
Experimento: 2 - Oversampling ({dataset_name})
Params RF: {trial.params}

Metricas Clave
F1 Macro (Validation): {f1_mac:.4f}

Classification Report (Precision, Recall, F1 por clase)
{report}
"""
        log_filename = f"{dir_name}/{dataset_name}_Trial_{trial.number}_Metrics.log"
        with open(log_filename, 'w', encoding='utf-8') as f:
            f.write(log_msg)
        
        print(f"[{dataset_name.upper()}] Trial {trial.number} completado | F1 Macro: {f1_mac:.4f}")
        
        return f1_mac

    study_name = f"RF_Exp2_{dataset_name}_Optimization"
    study_rf = optuna.create_study(direction='maximize', study_name=study_name)
    
    study_rf.optimize(objective_rf, n_trials=50) 
    
    print(f"\nResultados para: {dataset_name.upper()}")
    print(f"Mejor Trial: {study_rf.best_trial.number}")
    print(f"Mejor F1 Macro: {study_rf.best_value:.4f}")
    print(f"Mejores Hiperparametros:\n{study_rf.best_params}")
    
    with open(f"{dir_name}/Resumen_Exp2_Oversampling.txt", 'a', encoding='utf-8') as res_file:
        res_file.write(f"Dataset: {dataset_name.upper()}\n")
        res_file.write(f"Mejor F1 Macro: {study_rf.best_value:.4f} (Trial {study_rf.best_trial.number})\n")
        res_file.write(f"Params: {study_rf.best_params}\n")

print("Experimento 2 con sobremuestreo ha sido completado")

[I 2026-03-29 06:12:43,893] A new study created in memory with name: RF_Exp2_smote_Optimization


Experimento con Dataset: SMOTE


[I 2026-03-29 06:13:15,605] Trial 0 finished with value: 0.797813479481432 and parameters: {'n_estimators': 293, 'max_depth': 42, 'min_samples_split': 17, 'min_samples_leaf': 10, 'max_features': 'log2', 'criterion': 'entropy'}. Best is trial 0 with value: 0.797813479481432.


[SMOTE] Trial 0 completado | F1 Macro: 0.7978


[I 2026-03-29 06:13:48,894] Trial 1 finished with value: 0.7966737238808371 and parameters: {'n_estimators': 259, 'max_depth': 56, 'min_samples_split': 7, 'min_samples_leaf': 13, 'max_features': 'log2', 'criterion': 'gini'}. Best is trial 0 with value: 0.797813479481432.


...

[SMOTE_TOMEK] Trial 47 completado | F1 Macro: 0.7619


[I 2026-03-29 15:04:50,074] Trial 48 finished with value: 0.7954598190195383 and parameters: {'n_estimators': 383, 'max_depth': 29, 'min_samples_split': 6, 'min_samples_leaf': 4, 'max_features': None, 'criterion': 'entropy'}. Best is trial 20 with value: 0.8373884984106482.


[SMOTE_TOMEK] Trial 48 completado | F1 Macro: 0.7955


[I 2026-03-29 15:12:41,305] Trial 49 finished with value: 0.7463696365755158 and parameters: {'n_estimators': 442, 'max_depth': 19, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': None, 'criterion': 'gini'}. Best is trial 20 with value: 0.8373884984106482.


[SMOTE_TOMEK] Trial 49 completado | F1 Macro: 0.7464

Resultados para: SMOTE_TOMEK
Mejor Trial: 20
Mejor F1 Macro: 0.8374
Mejores Hiperparametros:
{'n_estimators': 141, 'max_depth': 76, 'min_samples_split': 18, 'min_samples_leaf': 5, 'max_features': None, 'criterion': 'gini'}
Experimento 2 con sobremuestreo ha sido completado


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
import optuna

dir_name = "Logs_RF_Baseline_3"
os.makedirs(dir_name, exist_ok=True)

X_train_raw, y_train_raw = train_datasets['none']

weight_methods = ['effective_samples', 'focal']

for method_name in weight_methods:
    print(f"Iniciando experimento 3 con funcion de pesos: {method_name.upper()}")
    
    def objective_rf(trial):
        if method_name == 'balanced':
            weight_dict = balanced_class_weight(y_train_raw)
        elif method_name == 'polynomial':
            alpha = trial.suggest_float('poly_alpha', 0.1, 1.0)
            weight_dict = polynomial_class_weight(y_train_raw, alpha=alpha)
        elif method_name == 'effective_samples':
            beta = trial.suggest_float('beta_eff_samp', 0.9, 0.9999, log=True)
            weight_dict = effective_samples_class_weight(y_train_raw, beta=beta)
        elif method_name == 'focal':
            gamma = trial.suggest_float('focal_gamma', 0.5, 5.0)
            weight_dict = focal_class_weight_improved(y_train_raw, gamma=gamma)
        
        rf_params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'max_depth': trial.suggest_int('max_depth', 10, 100),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
            'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
            'class_weight': weight_dict,
            'n_jobs': -1,
            'random_state': 42
        }
        
        model = RandomForestClassifier(**rf_params)
        model.fit(X_train_raw, y_train_raw)
        
        y_pred = model.predict(X_val)
        
        f1_mac = f1_score(y_val_1d, y_pred, average='macro')
        report = classification_report(y_val_1d, y_pred, zero_division=0)
        
        save_confusion_matrix(y_val_1d, y_pred, trial.number, method_name, phase="Val")
        
        log_msg = f"""Trial {trial.number}
Experimento: 3 - Class Weights ({method_name})
Params RF y Weights: {trial.params}

Metricas Clave
F1 Macro (Validation): {f1_mac:.4f}

Classification Report (Precision, Recall, F1 por clase)
{report}
"""
        log_filename = f"{dir_name}/{method_name}_Trial_{trial.number}_Metrics.log"
        with open(log_filename, 'w', encoding='utf-8') as f:
            f.write(log_msg)
        
        print(f"[{method_name.upper()}] Trial {trial.number} completado | F1 Macro: {f1_mac:.4f}")
        
        return f1_mac

    study_name = f"RF_Exp3_{method_name}_Optimization"
    study_rf = optuna.create_study(direction='maximize', study_name=study_name)
    
    study_rf.optimize(objective_rf, n_trials=50) 
    
    print(f"\nResultados para: {method_name.upper()}")
    print(f"Mejor Trial: {study_rf.best_trial.number}")
    print(f"Mejor F1 Macro: {study_rf.best_value:.4f}")
    print(f"Mejores Hiperparametros:\n{study_rf.best_params}")
    
    with open(f"{dir_name}/Resumen_Exp3_Weights.txt", 'a', encoding='utf-8') as res_file:
        res_file.write(f"Experimento Random Forest con Método de Peso: {method_name.upper()}\n")
        res_file.write(f"Mejor F1 Macro: {study_rf.best_value:.4f} (Trial {study_rf.best_trial.number})\n")
        res_file.write(f"Params: {study_rf.best_params}\n\n")

print("\nExperimento 3 con pesos completado")

[I 2026-03-30 01:52:56,824] A new study created in memory with name: RF_Exp3_effective_samples_Optimization


Iniciando experimento 3 con funcion de pesos: EFFECTIVE_SAMPLES


[I 2026-03-30 01:53:26,466] Trial 0 finished with value: 0.6846424455864963 and parameters: {'beta_eff_samp': 0.9711888657738788, 'n_estimators': 235, 'max_depth': 51, 'min_samples_split': 2, 'min_samples_leaf': 16, 'max_features': 'log2', 'criterion': 'gini'}. Best is trial 0 with value: 0.6846424455864963.


[EFFECTIVE_SAMPLES] Trial 0 completado | F1 Macro: 0.6846


[I 2026-03-30 02:01:32,205] Trial 1 finished with value: 0.8116955643382457 and parameters: {'beta_eff_samp': 0.9403170581130238, 'n_estimators': 360, 'max_depth': 30, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': None, 'criterion': 'gini'}. Best is trial 1 with value: 0.8116955643382457.


...

[FOCAL] Trial 47 completado | F1 Macro: 0.7444


[I 2026-03-30 05:58:31,683] Trial 48 finished with value: 0.761281778016162 and parameters: {'focal_gamma': 0.7124450633737294, 'n_estimators': 257, 'max_depth': 60, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': 'log2', 'criterion': 'entropy'}. Best is trial 43 with value: 0.8870735753069503.


[FOCAL] Trial 48 completado | F1 Macro: 0.7613


[I 2026-03-30 06:00:11,428] Trial 49 finished with value: 0.8866805919528804 and parameters: {'focal_gamma': 0.51414812425806, 'n_estimators': 220, 'max_depth': 53, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': None, 'criterion': 'entropy'}. Best is trial 43 with value: 0.8870735753069503.


[FOCAL] Trial 49 completado | F1 Macro: 0.8867

Resultados para: FOCAL
Mejor Trial: 43
Mejor F1 Macro: 0.8871
Mejores Hiperparametros:
{'focal_gamma': 1.388670032128727, 'n_estimators': 217, 'max_depth': 67, 'min_samples_split': 9, 'min_samples_leaf': 4, 'max_features': None, 'criterion': 'entropy'}

Experimento 3 con pesos completado


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.base import BaseEstimator, TransformerMixin
import optuna

dir_name = "Logs_RF_Baseline_4"
os.makedirs(dir_name, exist_ok=True)

X_train_raw, y_train_raw = train_datasets['none']

total_columns = X_train_raw.shape[1]

fs_methods = ['f_classif', 'tree']

for method in fs_methods:
    print(f"Experimento 4 con seleccion: {method.upper()}")
    
    def objective_rf(trial):
        k_features = trial.suggest_int('k_features', 30, total_columns)
        
        selector = FeatureSelector(method=method, k=k_features)
        X_train_fs = selector.fit_transform(X_train_raw, y_train_raw)
        X_val_fs = selector.transform(X_val)
        
        rf_params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'max_depth': trial.suggest_int('max_depth', 10, 100),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
            'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
            'n_jobs': -1,
            'random_state': 42
        }
        
        model = RandomForestClassifier(**rf_params)
        model.fit(X_train_fs, y_train_raw)
        
        y_pred = model.predict(X_val_fs)
        
        f1_mac = f1_score(y_val_1d, y_pred, average='macro')
        report = classification_report(y_val_1d, y_pred, zero_division=0)
        
        save_confusion_matrix(y_val_1d, y_pred, trial.number, method, phase="Val")
        
        log_msg = f"""Trial {trial.number}
Experimento: 4 - Feature Selection ({method})
Params FS (k): {k_features}
Params RF: {trial.params}

Metricas Clave
F1 Macro (Validation): {f1_mac:.4f}

Classification Report (Precision, Recall, F1 por clase)
{report}
"""
        log_filename = f"{dir_name}/{method}_Trial_{trial.number}_Metrics.log"
        with open(log_filename, 'w', encoding='utf-8') as f:
            f.write(log_msg)
        
        print(f"[{method.upper()} | k={k_features}] Trial {trial.number} completado | F1 Macro: {f1_mac:.4f}")
        
        return f1_mac

    study_name = f"RF_Exp4_FS_{method}_Optimization"
    study_rf = optuna.create_study(direction='maximize', study_name=study_name)
    
    study_rf.optimize(objective_rf, n_trials=50) 
    
    print(f"\nResultados para FS: {method.upper()}")
    print(f"Mejor Trial: {study_rf.best_trial.number}")
    print(f"Mejor F1 Macro: {study_rf.best_value:.4f}")
    print(f"Mejores Hiperparametros:\n{study_rf.best_params}")
    
    with open(f"{dir_name}/Resumen_Exp4_FS.txt", 'a', encoding='utf-8') as res_file:
        res_file.write(f"Método de Feature Selection: {method.upper()}\n")
        res_file.write(f"Mejor F1 Macro: {study_rf.best_value:.4f} (Trial {study_rf.best_trial.number})\n")
        res_file.write(f"Params: {study_rf.best_params}\n")

print("Experimento 4 con Feature Selection completado")

[I 2026-03-30 13:45:59,563] A new study created in memory with name: RF_Exp4_FS_tree_Optimization


Experimento 4 con seleccion: TREE


[I 2026-03-30 13:46:18,674] Trial 0 finished with value: 0.6854267258261159 and parameters: {'k_features': 51, 'n_estimators': 106, 'max_depth': 47, 'min_samples_split': 14, 'min_samples_leaf': 20, 'max_features': 'log2', 'criterion': 'entropy'}. Best is trial 0 with value: 0.6854267258261159.


[TREE | k=51] Trial 0 completado | F1 Macro: 0.6854


[I 2026-03-30 13:52:43,167] Trial 1 finished with value: 0.8236885679674373 and parameters: {'k_features': 50, 'n_estimators': 444, 'max_depth': 76, 'min_samples_split': 11, 'min_samples_leaf': 5, 'max_features': None, 'criterion': 'gini'}. Best is trial 1 with value: 0.8236885679674373.


...

[TREE | k=36] Trial 47 completado | F1 Macro: 0.7593


[I 2026-03-30 16:02:31,751] Trial 48 finished with value: 0.833387965461999 and parameters: {'k_features': 40, 'n_estimators': 250, 'max_depth': 66, 'min_samples_split': 20, 'min_samples_leaf': 8, 'max_features': None, 'criterion': 'entropy'}. Best is trial 33 with value: 0.8509815476378731.


[TREE | k=40] Trial 48 completado | F1 Macro: 0.8334


[I 2026-03-30 16:03:11,874] Trial 49 finished with value: 0.7335728014653824 and parameters: {'k_features': 42, 'n_estimators': 332, 'max_depth': 97, 'min_samples_split': 18, 'min_samples_leaf': 5, 'max_features': 'log2', 'criterion': 'entropy'}. Best is trial 33 with value: 0.8509815476378731.


[TREE | k=42] Trial 49 completado | F1 Macro: 0.7336

Resultados para FS: TREE
Mejor Trial: 33
Mejor F1 Macro: 0.8510
Mejores Hiperparametros:
{'k_features': 51, 'n_estimators': 453, 'max_depth': 77, 'min_samples_split': 17, 'min_samples_leaf': 2, 'max_features': None, 'criterion': 'entropy'}
Experimento 4 con Feature Selection completado


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
import optuna

dir_name = "Logs_RF_Baseline_6"
os.makedirs(dir_name, exist_ok=True)

def save_confusion_matrix(y_true, y_pred, trial_number, ds_name, fs_name, w_name, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'CM Final - Trial {trial_number}\nDS: {ds_name} | FS: {fs_name} | W: {w_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'{dir_name}/Trial_{trial_number}_Final_CM.png')
    plt.close()

def objective_final(trial):
    ds_name = trial.suggest_categorical('dataset', ['none', 'smote_enn'])
    X_train_raw, y_train_raw = train_datasets[ds_name]
    total_cols = X_train_raw.shape[1]

    fs_method = trial.suggest_categorical('fs_method', ['none', 'tree'])
    
    if fs_method == 'tree':
        k_opt = trial.suggest_int('k_features', 30, total_cols)
        selector = FeatureSelector(method='tree', k=k_opt)
        X_train_final = selector.fit_transform(X_train_raw, y_train_raw)
        X_val_final = selector.transform(X_val)
    else:
        k_opt = total_cols
        X_train_final = X_train_raw
        X_val_final = X_val

    w_method = trial.suggest_categorical('weight_method', ['none', 'balanced'])
    
    if w_method == 'balanced':
        weight_dict = balanced_class_weight(y_train_raw)
    else:
        weight_dict = None

    rf_params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 20, 100),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'class_weight': weight_dict,
        'n_jobs': -1,
        'random_state': 42
    }
    
    model = RandomForestClassifier(**rf_params)
    model.fit(X_train_final, y_train_raw)
    
    y_pred = model.predict(X_val_final)
    
    f1_mac = f1_score(y_val_1d, y_pred, average='macro')
    report = classification_report(y_val_1d, y_pred, zero_division=0)
    
    save_confusion_matrix(y_val_1d, y_pred, trial.number, ds_name, fs_method, w_method)
    
    log_msg = f"""Trial {trial.number} - Experimento 6 RF
Arquitectura: DS={ds_name} | FS={fs_method} (k={k_opt}) | Weights={w_method}
Params RF: {trial.params}

Metricas
F1 Macro: {f1_mac:.4f}
{report}
"""
    with open(f"{dir_name}/Trial_{trial.number}_Metrics.log", 'w', encoding='utf-8') as f:
        f.write(log_msg)
    
    print(f"Trial {trial.number} | F1: {f1_mac:.4f} | DS: {ds_name} | FS: {fs_method} | W: {w_method}")
    return f1_mac

study_final = optuna.create_study(direction='maximize', study_name="Experiment_Optimization_RF")

study_final.optimize(objective_final, n_trials=100)

print("Resultados Experimento 6 RF")
print(f"Mejor F1 Macro: {study_final.best_value:.4f}")
print(f"Mejor Configuración: {study_final.best_params}")

with open(f"{dir_name}/Informe_Experimento_6.txt", 'w', encoding='utf-8') as f:
    f.write("Resumen del Experimento 6\n")
    f.write(f"Mejor F1 Macro: {study_final.best_value:.4f}\n")
    f.write(f"Parámetros Ganadores:\n{study_final.best_params}\n")

[I 2026-03-30 23:03:52,037] A new study created in memory with name: Experiment_Optimization_RF
[I 2026-03-30 23:04:38,338] Trial 0 finished with value: 0.7941397381978379 and parameters: {'dataset': 'smote_enn', 'fs_method': 'tree', 'k_features': 46, 'weight_method': 'none', 'n_estimators': 372, 'max_depth': 77, 'min_samples_split': 12, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'criterion': 'entropy'}. Best is trial 0 with value: 0.7941397381978379.


Trial 0 | F1: 0.7941 | DS: smote_enn | FS: tree | W: none


[I 2026-03-30 23:05:10,146] Trial 1 finished with value: 0.7000027696684453 and parameters: {'dataset': 'none', 'fs_method': 'none', 'weight_method': 'none', 'n_estimators': 204, 'max_depth': 40, 'min_samples_split': 18, 'min_samples_leaf': 7, 'max_features': 'sqrt', 'criterion': 'gini'}. Best is trial 0 with value: 0.7941397381978379.


Trial 1 | F1: 0.7000 | DS: none | FS: none | W: none


[I 2026-03-30 23:06:20,299] Trial 2 finished with value: 0.7007961370771104 and parameters: {'dataset': 'none', 'fs_method': 'none', 'weight_method': 'none', 'n_estimators': 473, 'max_depth': 56, 'min_samples_split': 17, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'criterion': 'gini'}. Best is trial 0 with value: 0.7941397381978379.


...

Trial 97 | F1: 0.8787 | DS: none | FS: none | W: balanced


[I 2026-03-31 03:50:45,203] Trial 98 finished with value: 0.8762984705386683 and parameters: {'dataset': 'none', 'fs_method': 'none', 'weight_method': 'balanced', 'n_estimators': 357, 'max_depth': 30, 'min_samples_split': 6, 'min_samples_leaf': 5, 'max_features': None, 'criterion': 'entropy'}. Best is trial 87 with value: 0.8870476671360806.


Trial 98 | F1: 0.8763 | DS: none | FS: none | W: balanced


[I 2026-03-31 03:51:20,586] Trial 99 finished with value: 0.7944664004551715 and parameters: {'dataset': 'smote_enn', 'fs_method': 'none', 'weight_method': 'none', 'n_estimators': 328, 'max_depth': 31, 'min_samples_split': 7, 'min_samples_leaf': 6, 'max_features': 'log2', 'criterion': 'entropy'}. Best is trial 87 with value: 0.8870476671360806.


Trial 99 | F1: 0.7945 | DS: smote_enn | FS: none | W: none
Resultados Experimento 6 RF
Mejor F1 Macro: 0.8870
Mejor Configuración: {'dataset': 'none', 'fs_method': 'none', 'weight_method': 'balanced', 'n_estimators': 290, 'max_depth': 28, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'criterion': 'entropy'}


In [11]:
import joblib
X_train_none_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_none.joblib")
y_train_none_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_none.joblib")

X_train_smote_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_smote.joblib")
y_train_smote_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_smote.joblib")

X_train_smote_enn_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_smote_enn.joblib")
y_train_smote_enn_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_smote_enn.joblib")

X_train_smote_tomek_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_smote_tomek.joblib")
y_train_smote_tomek_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_smote_tomek.joblib")

X_val_imp_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/X_val_scaled_bayesian_grouped.pkl")
X_test_imp_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/X_test_scaled_bayesian_grouped.pkl")

y_val_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/y_val_grouped.pkl")
y_test_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/y_test_grouped.pkl")

In [12]:
train_datasets_grouped = {
    'none': (X_train_none_grouped, y_train_none_grouped.values.ravel() if isinstance(y_train_none_grouped, pd.DataFrame) else y_train_none_grouped.ravel()),
    'smote': (X_train_smote_grouped, y_train_smote_grouped.values.ravel() if isinstance(y_train_smote_grouped, pd.DataFrame) else y_train_smote_grouped.ravel()),
    'smote_enn': (X_train_smote_enn_grouped, y_train_smote_enn_grouped.values.ravel() if isinstance(y_train_smote_enn_grouped, pd.DataFrame) else y_train_smote_enn_grouped.ravel()),
    'smote_tomek': (X_train_smote_tomek_grouped, y_train_smote_tomek_grouped.values.ravel() if isinstance(y_train_smote_tomek_grouped, pd.DataFrame) else y_train_smote_tomek_grouped.ravel())
}

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
import optuna

dir_name = "Logs_RF_Baseline_7"
os.makedirs(dir_name, exist_ok=True)

def save_confusion_matrix(y_true, y_pred, trial_number, ds_name, fs_name, w_name, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'CM Final - Trial {trial_number}\nDS: {ds_name} | FS: {fs_name} | W: {w_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'{dir_name}/Trial_{trial_number}_Final_CM.png')
    plt.close()

def objective_final(trial):
    ds_name = trial.suggest_categorical('dataset', ['none', 'smote_enn'])
    X_train_raw, y_train_raw = train_datasets_grouped[ds_name]
    total_cols = X_train_raw.shape[1]

    fs_method = trial.suggest_categorical('fs_method', ['none', 'tree'])
    
    if fs_method == 'tree':
        k_opt = trial.suggest_int('k_features', 30, total_cols)
        selector = FeatureSelector(method='tree', k=k_opt)
        X_train_final = selector.fit_transform(X_train_raw, y_train_raw)
        X_val_final = selector.transform(X_val_imp_grouped)
    else:
        k_opt = total_cols
        X_train_final = X_train_raw
        X_val_final = X_val_imp_grouped

    w_method = trial.suggest_categorical('weight_method', ['none', 'balanced'])
    
    if w_method == 'balanced':
        weight_dict = balanced_class_weight(y_train_raw)
    else:
        weight_dict = None

    rf_params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 20, 100),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'class_weight': weight_dict,
        'n_jobs': -1,
        'random_state': 42
    }
    
    model = RandomForestClassifier(**rf_params)
    model.fit(X_train_final, y_train_raw)
    
    y_pred = model.predict(X_val_final)
    
    f1_mac = f1_score(y_val_grouped, y_pred, average='macro')
    report = classification_report(y_val_grouped, y_pred, zero_division=0)
    
    save_confusion_matrix(y_val_grouped, y_pred, trial.number, ds_name, fs_method, w_method)
    
    log_msg = f"""Trial {trial.number} - Experimento 7- Datos Agrupados
Arquitectura: DS={ds_name} | FS={fs_method} (k={k_opt}) | Weights={w_method}
Params RF: {trial.params}

Metricas
F1 Macro: {f1_mac:.4f}
{report}
"""
    with open(f"{dir_name}/Trial_{trial.number}_Metrics.log", 'w', encoding='utf-8') as f:
        f.write(log_msg)
    
    print(f"Trial {trial.number} | F1: {f1_mac:.4f} | DS: {ds_name} | FS: {fs_method} | W: {w_method}")
    return f1_mac

study_final = optuna.create_study(direction='maximize', study_name="Experiment_Optimization_RF")

study_final.optimize(objective_final, n_trials=100)

print("Resultados Experimento 7 RF")
print(f"Mejor F1 Macro: {study_final.best_value:.4f}")
print(f"Mejor Configuración: {study_final.best_params}")

with open(f"{dir_name}/Informe_Experimento_7.txt", 'w', encoding='utf-8') as f:
    f.write("Resumen del Experimento 7\n")
    f.write(f"Mejor F1 Macro: {study_final.best_value:.4f}\n")
    f.write(f"Parámetros Ganadores:\n{study_final.best_params}\n")

[I 2026-04-01 16:23:12,455] A new study created in memory with name: Experiment_Optimization_RF
[I 2026-04-01 16:26:15,374] Trial 0 finished with value: 0.9505971075413616 and parameters: {'dataset': 'smote_enn', 'fs_method': 'tree', 'k_features': 51, 'weight_method': 'balanced', 'n_estimators': 374, 'max_depth': 61, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': None, 'criterion': 'entropy'}. Best is trial 0 with value: 0.9505971075413616.


Trial 0 | F1: 0.9506 | DS: smote_enn | FS: tree | W: balanced


[I 2026-04-01 16:26:40,112] Trial 1 finished with value: 0.9524806991534314 and parameters: {'dataset': 'smote_enn', 'fs_method': 'none', 'weight_method': 'none', 'n_estimators': 199, 'max_depth': 30, 'min_samples_split': 4, 'min_samples_leaf': 5, 'max_features': 'log2', 'criterion': 'gini'}. Best is trial 1 with value: 0.9524806991534314.


Trial 1 | F1: 0.9525 | DS: smote_enn | FS: none | W: none


[I 2026-04-01 16:27:41,429] Trial 2 finished with value: 0.9451171560823035 and parameters: {'dataset': 'smote_enn', 'fs_method': 'tree', 'k_features': 63, 'weight_method': 'none', 'n_estimators': 550, 'max_depth': 72, 'min_samples_split': 9, 'min_samples_leaf': 8, 'max_features': 'log2', 'criterion': 'gini'}. Best is trial 1 with value: 0.9524806991534314.


...

Trial 97 | F1: 0.9174 | DS: none | FS: none | W: none


[I 2026-04-01 22:51:11,094] Trial 98 finished with value: 0.9673448625752961 and parameters: {'dataset': 'none', 'fs_method': 'none', 'weight_method': 'none', 'n_estimators': 447, 'max_depth': 76, 'min_samples_split': 18, 'min_samples_leaf': 3, 'max_features': None, 'criterion': 'entropy'}. Best is trial 85 with value: 0.9679104530231029.


Trial 98 | F1: 0.9673 | DS: none | FS: none | W: none


[I 2026-04-01 22:55:39,394] Trial 99 finished with value: 0.9493497938324897 and parameters: {'dataset': 'smote_enn', 'fs_method': 'none', 'weight_method': 'none', 'n_estimators': 345, 'max_depth': 90, 'min_samples_split': 17, 'min_samples_leaf': 1, 'max_features': None, 'criterion': 'entropy'}. Best is trial 85 with value: 0.9679104530231029.


Trial 99 | F1: 0.9493 | DS: smote_enn | FS: none | W: none
Resultados Experimento 7 RF
Mejor F1 Macro: 0.9679
Mejor Configuración: {'dataset': 'none', 'fs_method': 'none', 'weight_method': 'none', 'n_estimators': 409, 'max_depth': 74, 'min_samples_split': 19, 'min_samples_leaf': 2, 'max_features': None, 'criterion': 'entropy'}


In [18]:
#Pruebas en Set de Test

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix

log_folder_test = "Logs_Resultados_Test_Final_RF"
os.makedirs(log_folder_test, exist_ok=True)

print("Cargando datos de Test...")
X_test = pd.read_pickle("Sets_Post_Scaled_Imp/X_test_scaled_bayesian.pkl")

print("Preparando datos de entrenamiento...")
x_train_none, y_train_none = train_datasets['none']

best_params_rf = {
    'n_estimators': 314, 
    'max_depth': 83, 
    'min_samples_split': 5, 
    'min_samples_leaf': 2, 
    'max_features': None, 
    'criterion': 'entropy',
    'n_jobs': -1,
    'random_state': 42
}

print("Entrenando el modelo Random Forest con mejor baseline-Test 1")
final_rf_model = RandomForestClassifier(**best_params_rf)
final_rf_model.fit(x_train_none, y_train_none)

print("Generando predicciones en el Test Set...")
preds_test = final_rf_model.predict(X_test)

f1_mac_test = f1_score(y_test_1d, preds_test, average='macro')
report_test = classification_report(y_test_1d, preds_test, zero_division=0)

print(f"Resultados en Test Final-Baseline 1")
print(f"F1 Macro Alcanzado: {f1_mac_test:.4f}")
print("\nReporte de Clasificación:")
print(report_test)

cm = confusion_matrix(y_test_1d, preds_test)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'CM FINAL TEST - RF Baseline\nF1 Macro: {f1_mac_test:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{log_folder_test}/CM_Test_RF_Baseline.png')
plt.close()

with open(f"{log_folder_test}/Reporte_Test_RF_Baseline_Test_1.txt", "w", encoding='utf-8') as f:
    f.write("Evaluacion en Test-Experimento 1\n")
    f.write(f"Hiperparámetros de Optuna (Trial 52): {best_params_rf}\n\n")
    f.write(f"F1 Macro en Test: {f1_mac_test:.4f}\n\n")
    f.write("Reporte de Clasificación Detallado:\n")
    f.write(report_test)

print(f"\nPrueba finalizada exitosamente! Todo se ha guardado en '{log_folder_test}'")

Cargando datos de Test...
Preparando datos de entrenamiento...
Entrenando el modelo Random Forest con mejor baseline-Test 1


Generando predicciones en el Test Set...
Resultados en Test Final-Baseline 1
F1 Macro Alcanzado: 0.8527

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    307072
           1       0.76      0.53      0.62       295
           2       1.00      1.00      1.00     19204
           3       1.00      0.99      1.00      1544
           4       1.00      1.00      1.00     34661
           5       0.99      0.99      0.99       825
           6       0.99      1.00      1.00       870
           7       1.00      1.00      1.00      1190
           8       1.00      1.00      1.00         2
           9       1.00      1.00      1.00         5
          10       0.99      1.00      1.00     23840
          11       0.99      1.00      0.99       884
          12       0.74      0.80      0.77       226
          13       0.00      0.00      0.00         3
          14       0.48      0.40      0.43        98

   

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix

log_folder_test = "Logs_Resultados_Test_Final_RF"
os.makedirs(log_folder_test, exist_ok=True)

x_train_smote_enn, y_train_smote_enn = train_datasets['smote_enn']

best_params_rf_smote_enn = {
    'n_estimators': 200, 
    'max_depth': 80, 
    'min_samples_split': 14, 
    'min_samples_leaf': 2, 
    'max_features': None, 
    'criterion': 'gini',
    'n_jobs': -1,
    'random_state': 42
}

print("Entrenando el mejor modelo Random Forest del Experimento 2...")
final_rf_smote_enn = RandomForestClassifier(**best_params_rf_smote_enn)
final_rf_smote_enn.fit(x_train_smote_enn, y_train_smote_enn)

print("Generando predicciones en el Test Set...")
preds_test_smote_enn = final_rf_smote_enn.predict(X_test)

f1_mac_test = f1_score(y_test_1d, preds_test_smote_enn, average='macro')
report_test = classification_report(y_test_1d, preds_test_smote_enn, zero_division=0)

print(f"Resultados finales en Test - Experimento 2")
print(f"F1 Macro Alcanzado: {f1_mac_test:.4f}")
print("\nReporte de Clasificación:")
print(report_test)

cm = confusion_matrix(y_test_1d, preds_test_smote_enn)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'CM Final Test Set - RF Experimento 2\nF1 Macro: {f1_mac_test:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{log_folder_test}/CM_Test_RF_Baseline_Exp2.png')
plt.close()

with open(f"{log_folder_test}/Reporte_Test_RF_Baseline_Test_2.txt", "w", encoding='utf-8') as f:
    f.write("Evaluacion Final en Test Set - Experimento 2\n")
    f.write(f"Hiperparámetros de Optuna: {best_params_rf_smote_enn}\n\n")
    f.write(f"F1 Macro en Test: {f1_mac_test:.4f}\n\n")
    f.write("Reporte de Clasificación Detallado:\n")
    f.write(report_test)

print(f"\nPrueba finalizada exitosamente")

Entrenando el mejor modelo Random Forest del Experimento 2...


Generando predicciones en el Test Set...
Resultados finales en Test - Experimento 2
F1 Macro Alcanzado: 0.8447

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    307072
           1       0.38      0.81      0.52       295
           2       1.00      1.00      1.00     19204
           3       0.99      0.99      0.99      1544
           4       1.00      1.00      1.00     34661
           5       0.99      0.99      0.99       825
           6       1.00      0.99      0.99       870
           7       1.00      1.00      1.00      1190
           8       1.00      1.00      1.00         2
           9       1.00      0.80      0.89         5
          10       0.99      1.00      1.00     23840
          11       0.98      1.00      0.99       884
          12       0.72      0.73      0.73       226
          13       0.14      0.33      0.20         3
          14       0.36      0.40      0.38        

In [19]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix

log_folder_test = "Logs_Resultados_Test_Final_RF"
os.makedirs(log_folder_test, exist_ok=True)

x_train_none, y_train_none = train_datasets['none']

best_params_rf_exp3 = {
    'n_estimators': 219, 
    'max_depth': 30, 
    'min_samples_split': 7, 
    'min_samples_leaf': 1, 
    'max_features': None, 
    'criterion': 'entropy',
    'class_weight': 'balanced',
    'n_jobs': -1,
    'random_state': 42
}

print("Entrenando el mejor modelo Random Forest del Experimento 3 (Balanced)...")
final_rf_exp3 = RandomForestClassifier(**best_params_rf_exp3)
final_rf_exp3.fit(x_train_none, y_train_none)

print("Generando predicciones en el Test Set...")
preds_test_exp3 = final_rf_exp3.predict(X_test)

f1_mac_test = f1_score(y_test_1d, preds_test_exp3, average='macro')
report_test = classification_report(y_test_1d, preds_test_exp3, zero_division=0)

print(f"Resultados finales en Test - Experimento 3 (Pesos Balanced)")
print(f"F1 Macro Alcanzado: {f1_mac_test:.4f}")
print("\nReporte de Clasificación:")
print(report_test)

cm = confusion_matrix(y_test_1d, preds_test_exp3)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'CM Final Test Set - RF Experimento 3 (Balanced)\nF1 Macro: {f1_mac_test:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{log_folder_test}/CM_Test_RF_Baseline_Exp3.png')
plt.close()

with open(f"{log_folder_test}/Reporte_Test_RF_Baseline_Test_3.txt", "w", encoding='utf-8') as f:
    f.write("Evaluacion Final en Test Set - Experimento 3 (Pesos Balanced)\n")
    f.write(f"Hiperparámetros de Optuna: {best_params_rf_exp3}\n\n")
    f.write(f"F1 Macro en Test: {f1_mac_test:.4f}\n\n")
    f.write("Reporte de Clasificación Detallado:\n")
    f.write(report_test)

print(f"\nPrueba finalizada exitosamente en Test para Experimento 3")

Entrenando el mejor modelo Random Forest del Experimento 3 (Balanced)...
Generando predicciones en el Test Set...
Resultados finales en Test - Experimento 3 (Pesos Balanced)
F1 Macro Alcanzado: 0.8746

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    307072
           1       0.58      0.69      0.63       295
           2       1.00      1.00      1.00     19204
           3       0.99      1.00      1.00      1544
           4       1.00      1.00      1.00     34661
           5       0.99      0.99      0.99       825
           6       0.99      0.99      0.99       870
           7       1.00      1.00      1.00      1190
           8       1.00      1.00      1.00         2
           9       1.00      1.00      1.00         5
          10       0.99      1.00      1.00     23840
          11       0.98      1.00      0.99       884
          12       0.76      0.75      0.76       226
          13   

In [20]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix

log_folder_test = "Logs_Resultados_Test_Final_RF"
os.makedirs(log_folder_test, exist_ok=True)

x_train_none, y_train_none = train_datasets['none']

k_opt = 51
best_params_rf_exp4 = {
    'n_estimators': 453, 
    'max_depth': 77, 
    'min_samples_split': 17, 
    'min_samples_leaf': 2, 
    'max_features': None, 
    'criterion': 'entropy',
    'n_jobs': -1,
    'random_state': 42
}

print(f"Aplicando Feature Selection (Método: TREE, k={k_opt})...")
selector = FeatureSelector(method='tree', k=k_opt)
X_train_fs = selector.fit_transform(x_train_none, y_train_none)
X_test_fs = selector.transform(X_test) 

print("Entrenando el mejor modelo Random Forest del Experimento 4 (FS TREE)...")
final_rf_exp4 = RandomForestClassifier(**best_params_rf_exp4)
final_rf_exp4.fit(X_train_fs, y_train_none)

print("Generando predicciones en el Test Set...")
preds_test_exp4 = final_rf_exp4.predict(X_test_fs)

f1_mac_test = f1_score(y_test_1d, preds_test_exp4, average='macro')
report_test = classification_report(y_test_1d, preds_test_exp4, zero_division=0)

print(f"Resultados finales en Test - Experimento 4 (Feature Selection TREE)")
print(f"F1 Macro Alcanzado: {f1_mac_test:.4f}")
print("\nReporte de Clasificación:")
print(report_test)

cm = confusion_matrix(y_test_1d, preds_test_exp4)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'CM Final Test Set - RF Experimento 4 (FS TREE)\nF1 Macro: {f1_mac_test:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{log_folder_test}/CM_Test_RF_Baseline_Test_4.png')
plt.close()

with open(f"{log_folder_test}/Reporte_Test_RF_Baseline_Test_4.txt", "w", encoding='utf-8') as f:
    f.write("Evaluacion Final en Test Set - Experimento 4 (Feature Selection TREE)\n")
    f.write(f"k_features seleccionadas: {k_opt}\n")
    f.write(f"Hiperparámetros de Optuna: {best_params_rf_exp4}\n\n")
    f.write(f"F1 Macro en Test: {f1_mac_test:.4f}\n\n")
    f.write("Reporte de Clasificación Detallado:\n")
    f.write(report_test)

print(f"\nPrueba finalizada exitosamente en Test para Experimento 4")

Aplicando Feature Selection (Método: TREE, k=51)...
Entrenando el mejor modelo Random Forest del Experimento 4 (FS TREE)...
Generando predicciones en el Test Set...
Resultados finales en Test - Experimento 4 (Feature Selection TREE)
F1 Macro Alcanzado: 0.8536

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    307072
           1       0.84      0.50      0.63       295
           2       1.00      1.00      1.00     19204
           3       1.00      0.99      0.99      1544
           4       1.00      1.00      1.00     34661
           5       0.99      0.99      0.99       825
           6       1.00      0.99      1.00       870
           7       1.00      1.00      1.00      1190
           8       1.00      1.00      1.00         2
           9       1.00      1.00      1.00         5
          10       0.99      1.00      1.00     23840
          11       0.98      1.00      0.99       884
          

In [21]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix

log_folder_test = "Logs_Resultados_Test_Final_RF"
os.makedirs(log_folder_test, exist_ok=True)

x_train_none, y_train_none = train_datasets['none']

best_params_rf_exp6 = {
    'n_estimators': 290, 
    'max_depth': 28, 
    'min_samples_split': 5, 
    'min_samples_leaf': 5, 
    'max_features': None, 
    'criterion': 'entropy',
    'class_weight': 'balanced',
    'n_jobs': -1,
    'random_state': 42
}

print("Entrenando el mejor modelo Random Forest del Experimento 6...")
final_rf_exp6 = RandomForestClassifier(**best_params_rf_exp6)
final_rf_exp6.fit(x_train_none, y_train_none)

print("Generando predicciones en el Test Set...")
preds_test_exp6 = final_rf_exp6.predict(X_test)

f1_mac_test = f1_score(y_test_1d, preds_test_exp6, average='macro')
report_test = classification_report(y_test_1d, preds_test_exp6, zero_division=0)

print(f"Resultados finales en Test - Experimento 6")
print(f"F1 Macro Alcanzado: {f1_mac_test:.4f}")
print("\nReporte de Clasificación:")
print(report_test)

cm = confusion_matrix(y_test_1d, preds_test_exp6)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'CM Final Test Set - RF Experimento 6\nF1 Macro: {f1_mac_test:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{log_folder_test}/CM_Test_RF_Baseline_Exp6.png')
plt.close()

with open(f"{log_folder_test}/Reporte_Test_RF_Baseline_Test_6.txt", "w", encoding='utf-8') as f:
    f.write("Evaluacion Final en Test Set - Experimento 6\n")
    f.write(f"Configuración Ganadora: Dataset='none', FS='none', Weight='balanced'\n")
    f.write(f"Hiperparámetros de Optuna: {best_params_rf_exp6}\n\n")
    f.write(f"F1 Macro en Test: {f1_mac_test:.4f}\n\n")
    f.write("Reporte de Clasificación Detallado:\n")
    f.write(report_test)

print(f"\nPrueba finalizada exitosamente en Test para Experimento 6")

Entrenando el mejor modelo Random Forest del Experimento 6...
Generando predicciones en el Test Set...
Resultados finales en Test - Experimento 6
F1 Macro Alcanzado: 0.8608

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    307072
           1       0.54      0.80      0.64       295
           2       1.00      1.00      1.00     19204
           3       0.99      1.00      0.99      1544
           4       1.00      1.00      1.00     34661
           5       0.97      0.99      0.98       825
           6       0.99      1.00      0.99       870
           7       1.00      1.00      1.00      1190
           8       1.00      1.00      1.00         2
           9       0.83      1.00      0.91         5
          10       0.99      1.00      1.00     23840
          11       0.98      1.00      0.99       884
          12       0.76      0.72      0.74       226
          13       0.17      0.33      0.22

In [22]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix

log_folder_test = "Logs_Resultados_Test_Final_RF"
os.makedirs(log_folder_test, exist_ok=True)

x_train_none_grp, y_train_none_grp = train_datasets_grouped['none']

y_test_grp_1d = y_test_grouped.values.ravel() if isinstance(y_test_grouped, pd.DataFrame) else y_test_grouped.ravel()

best_params_rf_exp7 = {
    'n_estimators': 409, 
    'max_depth': 74, 
    'min_samples_split': 19, 
    'min_samples_leaf': 2, 
    'max_features': None, 
    'criterion': 'entropy',
    'n_jobs': -1,
    'random_state': 42
}

print("Entrenando el mejor modelo Random Forest del Experimento 7 (Datos Agrupados)...")
final_rf_exp7 = RandomForestClassifier(**best_params_rf_exp7)
final_rf_exp7.fit(x_train_none_grp, y_train_none_grp)

print("Generando predicciones en el Test Set agrupado...")
preds_test_exp7 = final_rf_exp7.predict(X_test_imp_grouped)

f1_mac_test = f1_score(y_test_grp_1d, preds_test_exp7, average='macro')
report_test = classification_report(y_test_grp_1d, preds_test_exp7, zero_division=0)

print(f"Resultados finales en Test - Experimento 7 (Datos Agrupados)")
print(f"F1 Macro Alcanzado: {f1_mac_test:.4f}")
print("\nReporte de Clasificación:")
print(report_test)

cm = confusion_matrix(y_test_grp_1d, preds_test_exp7)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'CM Final Test Set - RF Experimento 7 (Grouped)\nF1 Macro: {f1_mac_test:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{log_folder_test}/CM_Test_RF_Baseline_Exp7.png')
plt.close()

with open(f"{log_folder_test}/Reporte_Test_RF_Baseline_Test_7.txt", "w", encoding='utf-8') as f:
    f.write("Evaluacion Final en Test Set - Experimento 7 (Datos Agrupados)\n")
    f.write(f"Configuración Ganadora: Dataset='none', FS='none', Weight='none'\n")
    f.write(f"Hiperparámetros de Optuna: {best_params_rf_exp7}\n\n")
    f.write(f"F1 Macro en Test: {f1_mac_test:.4f}\n\n")
    f.write("Reporte de Clasificación Detallado:\n")
    f.write(report_test)

print(f"\nPrueba finalizada exitosamente en Test para Experimento 7")

<ipython-input-22-5c2f4bb077a3>:14: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  y_test_grp_1d = y_test_grouped.values.ravel() if isinstance(y_test_grouped, pd.DataFrame) else y_test_grouped.ravel()


Entrenando el mejor modelo Random Forest del Experimento 7 (Datos Agrupados)...
Generando predicciones en el Test Set agrupado...
Resultados finales en Test - Experimento 7 (Datos Agrupados)
F1 Macro Alcanzado: 0.9332

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    307072
           1       0.81      0.51      0.62       295
           2       1.00      1.00      1.00     19204
           3       1.00      0.99      1.00      1544
           4       1.00      1.00      1.00     34661
           5       0.99      0.99      0.99       825
           6       1.00      0.99      1.00       870
           7       1.00      1.00      1.00      1190
           8       1.00      1.00      1.00         2
           9       1.00      0.40      0.57         5
          10       0.99      1.00      1.00     23840
          11       0.98      1.00      0.99       884
          12       0.98      0.98      0.98       32